In [0]:
# ==========================================================
# PHASE 1: INITIALIZE RAW MASTER DATA
# Note: Deliberately injecting a duplicate and a null row 
# to test our data pipeline's cleaning logic.
# ==========================================================

raw_customer_records = [
    (201, "Vikram Singh", "Noida", 27, "Basic"),
    (202, "Neha Sharma", "Gurgaon", 29, "Premium"),
    (203, "Ravi Kumar", "Chandigarh", 32, "Basic"),
    (204, "Aditi Desai", "Pune", 25, "Elite"),
    # intentional duplicate row
    (204, "Aditi Desai", "Pune", 25, "Elite"),
    # intentional null value row
    (205, None, "Kolkata", 28, "Premium"),
    (206, "Saurabh Mishra", "Lucknow", 26, "Basic"),
    (207, "Kavita Reddy", "Hyderabad", 33, "Premium")
]

schema_cols = ["cust_id", "full_name", "location", "user_age", "tier"]

df_raw_customers = spark.createDataFrame(raw_customer_records, schema_cols)

print("--- Raw Master Dataset ---")
display(df_raw_customers)

--- Raw Master Dataset ---


cust_id,full_name,location,user_age,tier
201,Vikram Singh,Noida,27,Basic
202,Neha Sharma,Gurgaon,29,Premium
203,Ravi Kumar,Chandigarh,32,Basic
204,Aditi Desai,Pune,25,Elite
204,Aditi Desai,Pune,25,Elite
205,null,Kolkata,28,Premium
206,Saurabh Mishra,Lucknow,26,Basic
207,Kavita Reddy,Hyderabad,33,Premium


In [0]:
# ==========================================================
# PHASE 2: DATA CLEANSING
# Dropping duplicates and removing rows with missing names
# ==========================================================

df_cleaned = (
    df_raw_customers
    .dropDuplicates()
    .dropna(subset=["full_name"])
)

print("--- Cleansed Dataset ---")
display(df_cleaned)

--- Cleansed Dataset ---


cust_id,full_name,location,user_age,tier
201,Vikram Singh,Noida,27,Basic
202,Neha Sharma,Gurgaon,29,Premium
203,Ravi Kumar,Chandigarh,32,Basic
204,Aditi Desai,Pune,25,Elite
206,Saurabh Mishra,Lucknow,26,Basic
207,Kavita Reddy,Hyderabad,33,Premium


In [0]:
# ==========================================================
# PHASE 3: WRITE TO DELTA LAKE STORAGE
# ==========================================================

delta_table_name = "user_profiles_delta"

df_cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(delta_table_name)
    
print(f"✅ Successfully created Delta Table: {delta_table_name}")

✅ Successfully created Delta Table: user_profiles_delta


In [0]:
# ==========================================================
# PHASE 4: SIMULATE INCOMING DAILY INCREMENTAL DATA
# Includes state changes for existing users & new registrations
# ==========================================================

daily_updates = [
    # Update: Changed city & upgraded tier
    (202, "Neha Sharma", "Bangalore", 29, "Elite"),
    # Update: Age increased & upgraded tier
    (203, "Ravi Kumar", "Chandigarh", 33, "Premium"),
    # Insert: Brand new customer
    (208, "Manish Tiwari", "Delhi", 28, "Basic"),
    # Insert: Brand new customer
    (209, "Simran Kaur", "Amritsar", 24, "Premium")
]

df_incremental = spark.createDataFrame(daily_updates, schema_cols)

print("--- Incremental Payload ---")
display(df_incremental)

--- Incremental Payload ---


cust_id,full_name,location,user_age,tier
202,Neha Sharma,Bangalore,29,Elite
203,Ravi Kumar,Chandigarh,33,Premium
208,Manish Tiwari,Delhi,28,Basic
209,Simran Kaur,Amritsar,24,Premium


In [0]:
# ==========================================================
# PHASE 5 & 6: EXECUTE UPSERT (MERGE) OPERATION
# Leveraging DeltaTable API for SCD Type 1 Updates
# ==========================================================
from delta.tables import DeltaTable

target_table = DeltaTable.forName(spark, delta_table_name)

(target_table.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.cust_id = source.cust_id"
    )
    # Using 'All()' is a much cleaner way to write this than explicit mapping
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("✅ Merge executed successfully!")

✅ Merge executed successfully!


In [0]:
# ==========================================================
# PHASE 7 & VALIDATIONS
# Verifying row counts, duplicate checks, and null values
# ==========================================================
from pyspark.sql.functions import col

df_final_target = spark.table(delta_table_name)

print(f"Total Records After Merge: {df_final_target.count()}")

# Validation A: Duplicate Primary Key Check
print("--- Duplicate ID Check (Should be Empty) ---")
df_dupes = df_final_target.groupBy("cust_id").count().filter(col("count") > 1)
display(df_dupes)

# Validation B: Null Check
print("--- Null Name Check (Should be Empty) ---")
df_nulls = df_final_target.filter(col("full_name").isNull())
display(df_nulls)

print("--- Final Mutated Delta Table ---")
display(df_final_target.orderBy("cust_id"))

Total Records After Merge: 8
--- Duplicate ID Check (Should be Empty) ---


cust_id,count


--- Null Name Check (Should be Empty) ---


cust_id,full_name,location,user_age,tier


--- Final Mutated Delta Table ---


cust_id,full_name,location,user_age,tier
201,Vikram Singh,Noida,27,Basic
202,Neha Sharma,Bangalore,29,Elite
203,Ravi Kumar,Chandigarh,33,Premium
204,Aditi Desai,Pune,25,Elite
206,Saurabh Mishra,Lucknow,26,Basic
207,Kavita Reddy,Hyderabad,33,Premium
208,Manish Tiwari,Delhi,28,Basic
209,Simran Kaur,Amritsar,24,Premium


In [0]:
# ==========================================================
# EXPORT DATA TO CSV FOR GITHUB/LOCAL REPOSITORY
# ==========================================================

# Converting Spark DataFrames to Pandas to write standard CSVs
df_raw_customers.toPandas().to_csv("customer_master_records.csv", index=False)
df_incremental.toPandas().to_csv("customer_incremental_updates.csv", index=False)

print("CSV files generated successfully for repository upload.")

CSV files generated successfully for repository upload.
